In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_rows', None)

In [120]:
X_train = pd.read_csv('../../../data/processed/for_linear_model/MF,DF/predictors_train_filtered.csv')
X_test = pd.read_csv('../../../data/processed/for_linear_model/MF,DF/predictors_test_filtered.csv')
y_train = pd.read_csv('../../../data/processed/for_linear_model/MF,DF/target_train_filtered_logariphmed.csv')
y_test = pd.read_csv('../../../data/processed/for_linear_model/MF,DF/target_test_filtered_logariphmed.csv')

In [4]:
train_players = pd.read_csv('../../../data/processed/for_linear_model/MF,DF/train_players_names.csv')
test_players = pd.read_csv('../../../data/processed/for_linear_model/MF,DF/test_players_names.csv')

In [5]:
all_cols = X_train.columns
all_cols

Index(['Age', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
       '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG', 'Sh', 'SoT',
       'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'],
      dtype='object')

## Регрессор

In [122]:
LR_regressor = LinearRegression()
LR_regressor.fit(X_train, y_train)

LinearRegression()

## Предсказанные значения

In [123]:
y_pred_log = LR_regressor.predict(X_test)
y_pred = np.exp(y_pred_log)
y_test_original = np.exp(y_test)

y_train_pred = np.exp(LR_regressor.predict(X_train))
y_train_original = np.exp(y_train)

## Метрики

In [124]:
mae_test = mean_absolute_error(y_test_original, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2_test = r2_score(y_test_original, y_pred)
mape_test = mean_absolute_percentage_error(y_test_original, y_pred)

mae_train = mean_absolute_error(y_train_original, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train_original, y_train_pred))
r2_train = r2_score(y_train_original, y_train_pred)
mape_train = mean_absolute_percentage_error(y_train_original, y_train_pred)

metrics = pd.DataFrame({
    'MAE':[mae_test, mae_train],
    'RMSE':[rmse_test, rmse_train],
    'MAPE':[mape_test, mape_train],
    'R2':[r2_test, r2_train]
}, index = ['test', 'train'])

## Остатки

In [125]:
y_pred = pd.Series(np.reshape(y_pred, len(y_pred)), name='Value_pred')
y_train_pred = pd.Series(np.reshape(y_train_pred, len(y_train_pred)), name='Value_pred')

comp_leftovers_test = pd.concat([test_players, y_pred, y_test_original], axis=1)
comp_leftovers_train = pd.concat([train_players, y_train_pred, y_train_original], axis=1)

## Коэффициенты

In [126]:
coeffs = LR_regressor.coef_[0]
coeffs_df = pd.DataFrame({
    'coeff':coeffs.astype('float')
}, index=X_train.columns)
coeffs_df = coeffs_df.sort_values(by='coeff', key=abs, ascending=False)

# Все переменные

In [15]:
coeffs_df

,coeff
Min,1.543007e+01
90s,-1.434574e+01
Gls.1,2.157124e+00
G-PK.1,-2.124555e+00
Starts,-1.178213e+00
SoT,7.537068e-01
league_GB1,6.954386e-01
SoT/90,-5.794386e-01
Sh/90,5.585164e-01
league_IT1,4.317019e-01


In [17]:
metrics

,MAE,RMSE,MAPE,R2
test,4.971850,10.594509,0.780653,0.343797
train,4.600893,11.034440,0.741912,0.637514


# Потенциальные переменные для добавления и удаления

In [106]:
base_cols = ['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Crs',
        'Int', 'Sh', 'CrdY',
        'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']

## Для добавления:

In [108]:
cols_to_add = [col for col in all_cols if not(col in base_cols)]
new_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_add:
    X_train_new = X_train[base_cols + [col]]
    X_test_new = X_test[base_cols + [col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new<mae_test or mape_new<mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=new_cols.columns, index=[col])
        new_cols = pd.concat([new_cols, row])
new_cols

,mae,mape,r2
base,4.606361,0.604399,0.374934
G-PK,4.606262,1.917277,-0.064744
PK,4.606361,1.916851,-0.064762
PKatt,4.606361,1.916851,-0.064762
Gls.1,4.453901,1.956116,-0.033639
Ast.1,4.595683,1.911439,-0.059972
G-PK.1,4.493615,1.954362,-0.050180
Off,4.469131,1.918340,-0.099597
TklW,4.559147,1.839871,0.032167
OG,4.606361,1.916851,-0.064762


## Для удаления:

In [58]:
cols_to_delete = [col for col in base_cols if not(col.startswith('league'))]
del_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_delete:
    X_train_new = X_train[[c for c in base_cols if c != col]]
    X_test_new = X_test[[c for c in base_cols if c != col]]
    LR_regressor_new = LinearRegression()
    LR_regressor_new.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor_new.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new < mae_test or mape_new < mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=del_cols.columns, index=[col])
        del_cols = pd.concat([del_cols, row])
del_cols

,mae,mape,r2
base,4.283921,0.648172,0.418635
CrdY,4.225603,1.914135,-0.526563
Fls,4.256138,1.850532,-0.433993
SoT/90,4.231559,1.755173,-0.145306


# Модель 1

In [18]:
X_train = X_train[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
        'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG', 'Sh',
       'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
        'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG', 'Sh',
       'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [28]:
coeffs_df

,coeff
Gls.1,2.008133e+00
G-PK.1,-1.695967e+00
league_GB1,6.768671e-01
league_IT1,4.333538e-01
Ast,4.308539e-01
league_ES1,4.115579e-01
SoT/90,-3.680531e-01
Sh/90,3.551342e-01
league_L1,3.543621e-01
Sh,3.074069e-01


In [24]:
metrics

,MAE,RMSE,MAPE,R2
test,5.122085,10.822406,0.744841,0.315262
train,4.665983,10.971170,0.778829,0.641659


In [25]:
comp_leftovers_train

,Player,Value_pred,Value
0,Ilya Samoshnikov,3.365296,3.50
1,Federico Valverde,88.266096,130.00
2,Janik Haberer,1.768065,2.50
3,Andre Geraldes,0.805551,0.20
4,Mamadou Loum,0.835107,1.20
5,Sergio Carreira,5.786928,4.00
6,Ilnur Alshin,1.152953,0.50
7,Joe Aribo,10.573374,6.00
8,Edimilson Fernandes,1.484375,3.00
9,Nayel Mehssatou,2.196630,2.20


In [27]:
comp_leftovers_test

,Player,Value_pred,Value
0,Tomas Perez,0.525894,3.00
1,Warren Zaire-Emery,5.083613,55.00
2,Philipp Treu,4.940622,5.00
3,Desmond Acquah,0.463817,0.30
4,Oscar Mingueza,13.780557,20.00
5,Gabriel Moscardo,1.663593,8.00
6,Aleksandr Sandrachuk,0.780325,0.50
7,Jorthy Mokio,0.569431,8.00
8,James Sands,0.810549,3.00
9,Marvin Schulz,1.045815,0.70


# Модель 2

In [35]:
X_train = X_train[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK',
        'CrdR', 'CrdY', 'G+A.1','Fls', 'Fld', 'Off', 'Crs',
        'Int', 'TklW', 'Sh','SoT%', 'Sh/90', 'SoT/90', 'G/Sh',
        'G/SoT', 'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Gls', 'Ast', 'G+A', 'G-PK',
        'CrdR', 'CrdY', 'G+A.1','Fls', 'Fld', 'Off', 'Crs',
        'Int', 'TklW', 'Sh','SoT%', 'Sh/90', 'SoT/90', 'G/Sh',
        'G/SoT', 'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [46]:
coeffs_df

,coeff
league_GB1,0.720543
CrdY,-0.465623
league_ES1,0.441614
league_L1,0.428320
league_IT1,0.423664
Sh,0.412106
Fls,0.372594
SoT%,0.352575
Ast,0.304013
Sh/90,0.299887


In [26]:
metrics

,MAE,RMSE,MAPE,R2
test,5.122085,10.822406,0.744841,0.315262
train,4.665983,10.971170,0.778829,0.641659


In [13]:
comp_leftovers_train

,Player,Value_pred,Value
0,Ilya Samoshnikov,2.618553,3.50
1,Federico Valverde,70.503948,130.00
2,Janik Haberer,2.737228,2.50
3,Andre Geraldes,0.644619,0.20
4,Mamadou Loum,0.896633,1.20
5,Sergio Carreira,7.849105,4.00
6,Ilnur Alshin,1.600620,0.50
7,Joe Aribo,16.245647,6.00
8,Edimilson Fernandes,1.593523,3.00
9,Nayel Mehssatou,2.174515,2.20


In [14]:
comp_leftovers_test

,Player,Value_pred,Value
0,Tomas Perez,0.500229,3.00
1,Warren Zaire-Emery,6.643224,55.00
2,Philipp Treu,5.128795,5.00
3,Desmond Acquah,0.387419,0.30
4,Oscar Mingueza,12.219897,20.00
5,Gabriel Moscardo,1.096048,8.00
6,Aleksandr Sandrachuk,0.779859,0.50
7,Jorthy Mokio,0.982916,8.00
8,James Sands,0.901500,3.00
9,Marvin Schulz,0.871458,0.70


# Модель 3

In [45]:
X_train = X_train[['Age', '90s', 'Ast', 'G+A',
        'CrdY', 'G+A.1','Fls', 'Fld', 'Off', 'Crs',
        'Int', 'TklW', 'Sh', 'Sh/90', 'SoT/90',
        'G/SoT', 'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Ast', 'G+A',
        'CrdY', 'G+A.1','Fls', 'Fld', 'Off', 'Crs',
        'Int', 'TklW', 'Sh', 'Sh/90', 'SoT/90',
        'G/SoT', 'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [51]:
coeffs_df

,coeff
league_GB1,0.628319
CrdY,-0.409005
90s,0.400346
Ast,0.391792
league_ES1,0.381605
league_IT1,0.377260
league_L1,0.349202
Fls,0.313095
Crs,-0.300863
Sh,0.262610


In [52]:
metrics

,MAE,RMSE,MAPE,R2
test,4.283921,9.972083,0.648172,0.418635
train,4.977076,11.770965,0.848875,0.587509


In [53]:
comp_leftovers_train

,Player,Value_pred,Value
0,Ilya Samoshnikov,2.965257,3.50
1,Federico Valverde,70.924103,130.00
2,Janik Haberer,2.634147,2.50
3,Andre Geraldes,0.535035,0.20
4,Mamadou Loum,0.918454,1.20
5,Sergio Carreira,8.204141,4.00
6,Ilnur Alshin,1.507998,0.50
7,Joe Aribo,13.647999,6.00
8,Edimilson Fernandes,1.587880,3.00
9,Nayel Mehssatou,2.580102,2.20


In [54]:
comp_leftovers_test

,Player,Value_pred,Value
0,Tomas Perez,0.645499,3.00
1,Warren Zaire-Emery,6.533108,55.00
2,Philipp Treu,6.455020,5.00
3,Desmond Acquah,0.419046,0.30
4,Oscar Mingueza,15.074233,20.00
5,Gabriel Moscardo,1.089153,8.00
6,Aleksandr Sandrachuk,0.736784,0.50
7,Jorthy Mokio,0.969667,8.00
8,James Sands,1.126579,3.00
9,Marvin Schulz,0.663359,0.70


# Модель 4

In [68]:
X_train = X_train[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Off', 'Crs',
        'Int', 'TklW', 'Sh', 'Sh/90',
        'G/SoT', 'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Off', 'Crs',
        'Int', 'TklW', 'Sh', 'Sh/90',
        'G/SoT', 'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [74]:
coeffs_df

,coeff
league_GB1,0.618371
league_IT1,0.381162
league_ES1,0.377871
Ast,0.349301
league_L1,0.328785
G+A,0.255136
Crs,-0.246523
Sh/90,0.237200
Sh,0.235199
league_FR1,0.207111


In [75]:
metrics

,MAE,RMSE,MAPE,R2
test,4.228028,10.285888,0.686389,0.381471
train,4.676480,10.837535,0.845049,0.650336


In [76]:
comp_leftovers_train

,Player,Value_pred,Value
0,Ilya Samoshnikov,3.835788,3.50
1,Federico Valverde,85.528953,130.00
2,Janik Haberer,1.710462,2.50
3,Andre Geraldes,0.554349,0.20
4,Mamadou Loum,0.973474,1.20
5,Sergio Carreira,5.479852,4.00
6,Ilnur Alshin,1.177053,0.50
7,Joe Aribo,9.983952,6.00
8,Edimilson Fernandes,1.451505,3.00
9,Nayel Mehssatou,2.513479,2.20


In [77]:
comp_leftovers_test

,Player,Value_pred,Value
0,Tomas Perez,0.585302,3.00
1,Warren Zaire-Emery,5.137166,55.00
2,Philipp Treu,4.770309,5.00
3,Desmond Acquah,0.485517,0.30
4,Oscar Mingueza,19.901207,20.00
5,Gabriel Moscardo,1.540249,8.00
6,Aleksandr Sandrachuk,0.720507,0.50
7,Jorthy Mokio,1.019275,8.00
8,James Sands,0.952952,3.00
9,Marvin Schulz,0.862646,0.70


# Модель 5

In [86]:
X_train = X_train[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Off', 'Crs',
        'Int', 'TklW', 'Sh', 'Sh/90', 'CrdY',
        'G/SoT', 'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Off', 'Crs',
        'Int', 'TklW', 'Sh', 'Sh/90', 'CrdY',
        'G/SoT', 'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [92]:
coeffs_df

,coeff
league_GB1,0.627912
90s,0.409147
CrdY,-0.405187
league_ES1,0.376781
Ast,0.375582
league_IT1,0.373878
league_L1,0.342837
Fls,0.307631
Crs,-0.297349
Sh,0.245874


In [93]:
metrics

,MAE,RMSE,MAPE,R2
test,4.231559,9.943165,0.648996,0.422002
train,4.982563,11.810597,0.847874,0.584727


In [94]:
comp_leftovers_train

,Player,Value_pred,Value
0,Ilya Samoshnikov,3.040318,3.50
1,Federico Valverde,69.178032,130.00
2,Janik Haberer,2.602067,2.50
3,Andre Geraldes,0.531200,0.20
4,Mamadou Loum,0.954562,1.20
5,Sergio Carreira,8.200330,4.00
6,Ilnur Alshin,1.521022,0.50
7,Joe Aribo,13.784577,6.00
8,Edimilson Fernandes,1.629517,3.00
9,Nayel Mehssatou,2.525121,2.20


In [95]:
comp_leftovers_test

,Player,Value_pred,Value
0,Tomas Perez,0.636176,3.00
1,Warren Zaire-Emery,6.540751,55.00
2,Philipp Treu,6.469560,5.00
3,Desmond Acquah,0.426085,0.30
4,Oscar Mingueza,15.356645,20.00
5,Gabriel Moscardo,1.254621,8.00
6,Aleksandr Sandrachuk,0.721872,0.50
7,Jorthy Mokio,0.988178,8.00
8,James Sands,1.117833,3.00
9,Marvin Schulz,0.653479,0.70


# Модель 6

In [96]:
X_train = X_train[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Crs',
        'Int', 'Sh', 'CrdY',
        'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Crs',
        'Int', 'Sh', 'CrdY',
        'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [102]:
coeffs_df

,coeff
league_GB1,0.685622
Sh,0.485414
league_IT1,0.416379
league_ES1,0.410039
league_L1,0.405901
CrdY,-0.358037
Crs,-0.293371
Ast,0.291118
Fls,0.245688
league_PO1,0.245375


In [103]:
metrics

,MAE,RMSE,MAPE,R2
test,4.606361,10.340092,0.604399,0.374934
train,4.865338,11.395057,0.850999,0.613434


In [104]:
comp_leftovers_train

,Player,Value_pred,Value
0,Ilya Samoshnikov,3.162465,3.50
1,Federico Valverde,83.526757,130.00
2,Janik Haberer,3.056207,2.50
3,Andre Geraldes,0.549679,0.20
4,Mamadou Loum,0.849636,1.20
5,Sergio Carreira,7.650201,4.00
6,Ilnur Alshin,1.427921,0.50
7,Joe Aribo,15.129429,6.00
8,Edimilson Fernandes,1.602753,3.00
9,Nayel Mehssatou,2.244166,2.20


In [105]:
comp_leftovers_test

,Player,Value_pred,Value
0,Tomas Perez,0.802514,3.00
1,Warren Zaire-Emery,5.942096,55.00
2,Philipp Treu,6.235929,5.00
3,Desmond Acquah,0.395730,0.30
4,Oscar Mingueza,14.081132,20.00
5,Gabriel Moscardo,0.878594,8.00
6,Aleksandr Sandrachuk,0.630826,0.50
7,Jorthy Mokio,0.813257,8.00
8,James Sands,1.448511,3.00
9,Marvin Schulz,0.765004,0.70


# Модель 7

In [110]:
X_train = X_train[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Crs',
        'Int', 'Sh', 'CrdY', 'TklW', 'G/SoT',
        'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Crs',
        'Int', 'Sh', 'CrdY', 'TklW', 'G/SoT',
        'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [116]:
coeffs_df

,coeff
league_GB1,0.691533
Sh,0.521531
Ast,0.415302
CrdY,-0.405899
league_IT1,0.398249
league_ES1,0.396696
league_L1,0.394910
Fls,0.326552
Crs,-0.312627
90s,0.245604


In [117]:
metrics

,MAE,RMSE,MAPE,R2
test,4.550414,10.218123,0.633095,0.389594
train,5.011802,11.887067,0.855382,0.579332


In [118]:
comp_leftovers_train

,Player,Value_pred,Value
0,Ilya Samoshnikov,3.139587,3.50
1,Federico Valverde,72.433412,130.00
2,Janik Haberer,2.993041,2.50
3,Andre Geraldes,0.534490,0.20
4,Mamadou Loum,0.837587,1.20
5,Sergio Carreira,8.483575,4.00
6,Ilnur Alshin,1.541724,0.50
7,Joe Aribo,14.872709,6.00
8,Edimilson Fernandes,1.690339,3.00
9,Nayel Mehssatou,2.516180,2.20


In [119]:
comp_leftovers_test

,Player,Value_pred,Value
0,Tomas Perez,0.796853,3.00
1,Warren Zaire-Emery,6.601790,55.00
2,Philipp Treu,6.060081,5.00
3,Desmond Acquah,0.378087,0.30
4,Oscar Mingueza,13.854026,20.00
5,Gabriel Moscardo,0.869940,8.00
6,Aleksandr Sandrachuk,0.652062,0.50
7,Jorthy Mokio,1.056697,8.00
8,James Sands,1.438282,3.00
9,Marvin Schulz,0.706866,0.70


# Модель 8

In [121]:
X_train = X_train[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Crs', 'Off',
        'Int', 'Sh', 'CrdY', 'TklW', 'G/SoT', 'SoT/90',
        'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['Age', '90s', 'Ast', 'G+A',
        'G+A.1','Fls', 'Fld', 'Crs', 'Off',
        'Int', 'Sh', 'CrdY', 'TklW', 'G/SoT', 'SoT/90',
        'league_ES1', 'league_FR1','league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [127]:
coeffs_df

,coeff
league_GB1,0.652205
Ast,0.445104
CrdY,-0.418056
Sh,0.405216
league_ES1,0.401035
league_IT1,0.393974
league_L1,0.382905
Fls,0.332998
90s,0.316926
Crs,-0.311276


In [128]:
metrics

,MAE,RMSE,MAPE,R2
test,4.402375,10.113996,0.637125,0.401971
train,5.008232,11.703111,0.853053,0.592251
